# Analyse RCEMIP — Simulation `large300`
## Transport vertical de la quantité de mouvement par la turbulence convective

**Objectifs :**
1. Chargement robuste (lazy, chunked) des fichiers 3D volumieux
2. Détection de l'auto-agrégation convective via la PRW (fichier 2D natif)
3. Classification sec / humide et cartes 2D XY
4. Bilan de quantité de mouvement global et conditionnel (par régime turbulent)
5. Densité ρ₀ calculée à partir de la pression et de la température virtuelle


## 1. Librairies et constantes physiques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import os

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

# Constantes physiques
Rd      = 287.05   # J kg⁻¹ K⁻¹  — constante gaz sec
Rv      = 461.5    # J kg⁻¹ K⁻¹  — constante vapeur d'eau
EPSILON = Rd / Rv  # ≈ 0.622

print(f'Rd = {Rd} J/kg/K  |  Rv = {Rv} J/kg/K  |  ε = Rd/Rv = {EPSILON:.4f}')


## 2. Chargement des données

Les fichiers 3D (300×300×74×100 points) sont ouverts en mode **lazy** avec chunks Dask :  
rien n'est chargé en RAM à ce stade. Les calculs se feront par **blocs temporels**  
pour ne jamais dépasser une fraction de la mémoire disponible.

La PRW est lue directement depuis son fichier 2D natif, ce qui permet d'avoir  
la résolution temporelle et spatiale complète sans aucun recalcul.


In [ ]:
DIR_3D = '3D'
DIR_2D = '2D'
DIR_1D = '1D'

# Taille des blocs temporels pour les boucles de calcul.
# Réduire à 5 si la RAM est inférieure à 16 Go.
BLOC = 10

def open_lazy(repertoire, varname):
    """Ouvre un fichier NetCDF RCEMIP en mode lazy avec chunks sur le temps."""
    prefix = os.path.basename(repertoire.rstrip('/'))
    # Convention de nommage : MESONH_RCE_large300_<dim>_<var>.nc
    path = os.path.join(repertoire, f'MESONH_RCE_large300_{prefix}_{varname}.nc')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Fichier introuvable : {path}')
    return xr.open_dataset(path, chunks={'time': BLOC})

# --- Fichiers 3D (lazy) ---
ds_ua  = open_lazy(DIR_3D, 'ua')   # vent zonal u          [m/s]
ds_va  = open_lazy(DIR_3D, 'va')   # vent méridien v       [m/s]
ds_wa  = open_lazy(DIR_3D, 'wa')   # vitesse verticale w   [m/s]
ds_ta  = open_lazy(DIR_3D, 'ta')   # température T         [K]
ds_pa  = open_lazy(DIR_3D, 'pa')   # pression p            [Pa]
ds_hus = open_lazy(DIR_3D, 'hus')  # humidité spécifique   [kg/kg]
ds_clw = open_lazy(DIR_3D, 'clw')  # eau liquide nuageuse  [kg/kg]
ds_cli = open_lazy(DIR_3D, 'cli')  # glace nuageuse        [kg/kg]

# --- Fichier 2D natif de la PRW (chargement complet : léger) ---
ds_prw = open_lazy(DIR_2D, 'prw')  # eau précipitable      [kg/m²]

# --- DataArrays ---
u   = ds_ua['ua']
v   = ds_va['va']
w   = ds_wa['wa']
T   = ds_ta['ta']
p   = ds_pa['pa']
qv  = ds_hus['hus']
clw = ds_clw['clw']
cli = ds_cli['cli']
prw = ds_prw['prw']  # (time, y, x)

# --- Axes de coordonnées ---
dim_t = u.dims[0]   # 'time'
dim_z = u.dims[1]   # 'altitude'
dim_y = u.dims[2]   # 'y'
dim_x = u.dims[3]   # 'x'
alt   = u[dim_z].values  # vecteur altitude [m]
n_times = u.sizes[dim_t]

# Indice de début de l'état stationnaire : dernier tiers de la simulation
t_stat = int(2 * n_times / 3)
idx_stat = slice(t_stat, None)

print('Données ouvertes en mode lazy.')
print(f'  Grille 3D : {dict(u.sizes)}')
print(f'  Grille 2D PRW : {dict(prw.sizes)}')
print(f'  Axes : t={dim_t}, z={dim_z}, y={dim_y}, x={dim_x}')
print(f'  État stationnaire : pas de temps {t_stat} → {n_times-1}')


## 3. Profil de densité de référence ρ₀

ρ₀ est calculé à partir de la loi des gaz parfaits appliquée à l'air humide :
$$\rho_0 = \frac{p}{R_d \, T_v}, \qquad T_v = T \cdot \frac{1 + q_v/\varepsilon}{1 + q_v}$$
On moyenne ρ₀ sur le domaine horizontal et sur l'état stationnaire  
pour obtenir un profil vertical de référence 1D.


In [ ]:
# Calcul par blocs sur l'état stationnaire
rho0_sum = np.zeros(len(alt))
n_rho = 0

for t0 in range(t_stat, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    T_blk  = T.isel({dim_t: sl}).load()   # (bloc, z, y, x)
    p_blk  = p.isel({dim_t: sl}).load()
    qv_blk = qv.isel({dim_t: sl}).load()

    Tv_blk  = T_blk * (1.0 + qv_blk / EPSILON) / (1.0 + qv_blk)
    rho_blk = p_blk / (Rd * Tv_blk)

    rho0_sum += rho_blk.mean(dim=[dim_t, dim_y, dim_x]).values * (t1 - t0)
    n_rho    += (t1 - t0)

rho0_np = rho0_sum / n_rho   # profil 1D numpy [kg/m³]

print(f'ρ₀ calculé sur {n_rho} pas de temps.')
print(f'  Surface  : {rho0_np[0]:.3f} kg/m³')
print(f'  ~10 km   : {rho0_np[np.argmin(np.abs(alt - 10000))]:.3f} kg/m³')

fig, ax = plt.subplots(figsize=(4, 6))
ax.plot(rho0_np, alt, color='steelblue', lw=2)
ax.set_xlabel('ρ₀ (kg/m³)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Profil vertical ρ₀', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Eau précipitable (PRW) et auto-agrégation

La PRW est lue directement depuis le fichier 2D natif du modèle :  
on dispose ainsi de la **résolution temporelle et spatiale complète**,  
sans approximation de recalcul.

Le signe d'**auto-agrégation convective** est une distribution bimodale de la PRW :  
des colonnes très sèches coexistent avec des colonnes très humides.


In [ ]:
# Chargement complet de la PRW sur l'état stationnaire
# (fichier 2D : léger, pas de problème mémoire)
prw_stat = prw.isel({dim_t: idx_stat}).load()  # (t_stat, y, x)

# Moyenne temporelle → carte 2D de référence
prw_mean = prw_stat.mean(dim=dim_t)  # (y, x)

print(f'PRW chargée. Plage : {float(prw_mean.min()):.1f} — {float(prw_mean.max()):.1f} kg/m²')
print(f'Médiane : {float(np.median(prw_mean.values)):.1f} kg/m²')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Carte 2D ---
ax = axes[0]
im = ax.pcolormesh(prw_mean[dim_x].values, prw_mean[dim_y].values,
                   prw_mean.values, cmap='Blues', vmin=0)
cb = fig.colorbar(im, ax=ax)
cb.set_label('PRW (kg/m²)')
ax.set_title('Carte PRW moyenne — état stationnaire', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal')

# --- Distribution ---
ax2 = axes[1]
prw_flat = prw_mean.values.ravel()
med = np.median(prw_flat)
ax2.hist(prw_flat, bins=80, color='steelblue', edgecolor='white', lw=0.3)
ax2.axvline(med, color='red', lw=1.5, linestyle='--',
            label=f'Médiane = {med:.1f} kg/m²')
ax2.set_xlabel('PRW (kg/m²)')
ax2.set_ylabel('Nombre de colonnes')
ax2.set_title('Distribution de la PRW\n'
              '(bimodale → auto-agrégation)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Évolution temporelle de la variance de PRW
# La variance augmente si l'agrégation se développe au cours de la simulation
prw_all = prw.load()  # (time, y, x)  — fichier 2D : chargement total OK

prw_var  = prw_all.var(dim=[dim_y, dim_x])   # variance spatiale à chaque t
prw_mean_t = prw_all.mean(dim=[dim_y, dim_x]) # moyenne spatiale à chaque t

time_days = prw_all[dim_t].values.astype('float64') / (1e9 * 3600 * 24)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(time_days, prw_mean_t.values, color='steelblue', lw=1.5)
ax1.axvline(time_days[t_stat], color='red', lw=1, linestyle='--',
            label='Début état stationnaire')
ax1.set_ylabel('PRW moyenne (kg/m²)')
ax1.set_title('Évolution temporelle de la PRW', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(time_days, prw_var.values, color='darkorange', lw=1.5)
ax2.axvline(time_days[t_stat], color='red', lw=1, linestyle='--')
ax2.set_ylabel('Variance spatiale PRW (kg²/m⁴)')
ax2.set_xlabel('Temps (jours)')
ax2.set_title('Variance spatiale de PRW — signal d\'agrégation', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Classification sec / humide

On calibre le seuil sur la distribution de PRW observée au §4.  
**Modifier `PRW_SEUIL`** si la distribution est bimodale et que le creux  
entre les deux modes est visible à une valeur différente de la médiane.


In [ ]:
# Seuil initial = médiane (à ajuster selon la distribution)
PRW_SEUIL = float(np.median(prw_mean.values))
print(f'Seuil PRW retenu : {PRW_SEUIL:.1f} kg/m²')
print('→ Modifier PRW_SEUIL si le creux de la distribution bimodale est ailleurs.')

masque_humide = prw_mean > PRW_SEUIL   # DataArray bool (y, x)
masque_sec    = ~masque_humide

mh = masque_humide.values  # numpy bool (y, x) — utilisé dans les boucles
ms = masque_sec.values

frac_h = mh.mean() * 100
frac_s = ms.mean() * 100
print(f'Fraction humide : {frac_h:.1f}%  |  Fraction sèche : {frac_s:.1f}%')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Carte sec / humide ---
ax = axes[0]
cmap_sh = mcolors.ListedColormap(['#f5deb3', '#1a6faf'])  # sec=sable, humide=bleu
im = ax.pcolormesh(masque_humide[dim_x].values, masque_humide[dim_y].values,
                   masque_humide.values.astype(int), cmap=cmap_sh, vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax, ticks=[0.25, 0.75])
cbar.ax.set_yticklabels(['Sec', 'Humide'])
ax.set_title(f'Carte sec / humide  (seuil PRW = {PRW_SEUIL:.1f} kg/m²)',
             fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal')

# --- Histogramme avec seuil ---
ax2 = axes[1]
ax2.hist(prw_flat, bins=80, color='steelblue', edgecolor='white', lw=0.3)
ax2.axvline(PRW_SEUIL, color='black', lw=2, linestyle='--',
            label=f'Seuil = {PRW_SEUIL:.1f} kg/m²')
ax2.axvspan(0, PRW_SEUIL, alpha=0.08, color='saddlebrown', label='Sec')
ax2.axvspan(PRW_SEUIL, prw_flat.max(), alpha=0.08, color='royalblue', label='Humide')
ax2.set_xlabel('PRW (kg/m²)')
ax2.set_ylabel('Nombre de colonnes')
ax2.set_title('Distribution PRW avec seuil sec/humide', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Profil vertical du vent horizontal moyen

In [ ]:
def profil_vent(idx_slice, titre):
    """Calcule et trace le profil vertical du vent moyen sur une tranche temporelle."""
    u_m = u.isel({dim_t: idx_slice}).mean(dim=[dim_t, dim_y, dim_x]).compute()
    v_m = v.isel({dim_t: idx_slice}).mean(dim=[dim_t, dim_y, dim_x]).compute()
    mod = np.sqrt(u_m**2 + v_m**2)

    fig, ax = plt.subplots(figsize=(5, 7))
    ax.plot(u_m.values,  alt, color='royalblue', lw=2, label=r'$\overline{u}$')
    ax.plot(v_m.values,  alt, color='seagreen',  lw=2, label=r'$\overline{v}$')
    ax.plot(mod.values,  alt, 'k--', lw=1.5,         label='Module')
    ax.axvline(0, color='red', alpha=0.4, lw=1)
    ax.set_xlabel('Vitesse (m/s)')
    ax.set_ylabel('Altitude (m)')
    ax.set_title(titre, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

profil_vent(slice(0, 5),    'Profil du vent — état initial')
profil_vent(idx_stat,       'Profil du vent — état stationnaire')


## 7. Bilan de quantité de mouvement — global

L'équation de QdM zonale moyennée horizontalement :
$$\frac{\partial \overline{u}}{\partial t} =
-\frac{1}{\rho_0}\frac{\partial}{\partial z}\bigl(\rho_0\,\overline{u'w'}\bigr)
+ \text{advection} + \text{pression} + \text{rappel}$$

On calcule ici le **terme de Reynolds** : le flux ρ₀⟨u′w′⟩ et sa divergence verticale.


In [ ]:
# ======================================================================
# Calcul du flux ρ₀ <u'w'> — boucle par blocs temporels
# ======================================================================
flux_uw_sum = np.zeros(len(alt))
flux_vw_sum = np.zeros(len(alt))
n_glob = 0

for t0 in range(t_stat, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    u_blk = u.isel({dim_t: sl}).load()
    v_blk = v.isel({dim_t: sl}).load()
    w_blk = w.isel({dim_t: sl}).load()

    # Décomposition de Reynolds : anomalie = valeur - moyenne horizontale
    u_p = u_blk - u_blk.mean(dim=[dim_y, dim_x])
    v_p = v_blk - v_blk.mean(dim=[dim_y, dim_x])
    w_p = w_blk - w_blk.mean(dim=[dim_y, dim_x])

    # ρ₀ est un profil 1D (z) → le broadcast se fait automatiquement
    rho0_da = xr.DataArray(rho0_np, dims=[dim_z], coords={dim_z: u[dim_z]})

    flux_uw_blk = (rho0_da * u_p * w_p).mean(dim=[dim_t, dim_y, dim_x]).values
    flux_vw_blk = (rho0_da * v_p * w_p).mean(dim=[dim_t, dim_y, dim_x]).values

    n_bloc = t1 - t0
    flux_uw_sum += flux_uw_blk * n_bloc
    flux_vw_sum += flux_vw_blk * n_bloc
    n_glob      += n_bloc
    print(f'  t={t0:03d}–{t1-1:03d}')

flux_uw = flux_uw_sum / n_glob  # ρ₀ <u'w'>  [kg m⁻¹ s⁻²]
flux_vw = flux_vw_sum / n_glob

print(f'\nFlux globaux calculés sur {n_glob} pas de temps.')


In [ ]:
# Tendance : -1/ρ₀ · ∂(ρ₀ <u'w'>) / ∂z
def tendance(flux_profil):
    """Divergence verticale du flux pondéré, divisée par ρ₀."""
    return -np.gradient(flux_profil, alt) / rho0_np

tend_u_glob = tendance(flux_uw)
tend_v_glob = tendance(flux_vw)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

ax1.plot(flux_uw, alt, color='royalblue', lw=2, label=r"$\rho_0 \overline{u'w'}$")
ax1.plot(flux_vw, alt, color='seagreen',  lw=2, linestyle='--',
         label=r"$\rho_0 \overline{v'w'}$")
ax1.axvline(0, color='grey', alpha=0.5)
ax1.set_xlabel('Flux ρ₀⟨φ′w′⟩  (kg m⁻¹ s⁻²)')
ax1.set_ylabel('Altitude (m)')
ax1.set_title('A. Flux vertical de QdM', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(tend_u_glob * 1e5, alt, color='royalblue', lw=2, label='Tendance u')
ax2.plot(tend_v_glob * 1e5, alt, color='seagreen',  lw=2, linestyle='--',
         label='Tendance v')
ax2.axvline(0, color='grey', alpha=0.5)
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.set_title('B. Tendance du vent par la turbulence\n'
              r'$-\frac{1}{\rho_0}\frac{\partial}{\partial z}'
              r'(\rho_0\overline{\phi\'w\'})$', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan global de QdM — état stationnaire', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Bilan conditionnel — décomposition par régime turbulent

Le flux total ρ₀⟨u′w′⟩ est décomposé en quatre contributions physiques :

| Régime | Condition |
|--------|-----------|
| Convection nuageuse | w′ > +SEUIL_W  **et**  qc > SEUIL_QC |
| Thermiques secs (CBL) | w′ > +SEUIL_W  **et**  qc ≤ SEUIL_QC |
| Subsidence | w′ < −SEUIL_W |
| Petite turbulence | résidu |


In [ ]:
SEUIL_W  = 0.05   # m/s   — updraft / downdraft significatif
SEUIL_QC = 1e-5   # kg/kg — présence de condensats nuageux

flux_conv_sum = np.zeros(len(alt))
flux_sec_sum  = np.zeros(len(alt))
flux_sub_sum  = np.zeros(len(alt))
flux_tot2_sum = np.zeros(len(alt))
n_cond = 0

rho0_da = xr.DataArray(rho0_np, dims=[dim_z], coords={dim_z: u[dim_z]})

for t0 in range(t_stat, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    u_blk   = u.isel({dim_t: sl}).load()
    w_blk   = w.isel({dim_t: sl}).load()
    clw_blk = clw.isel({dim_t: sl}).load()
    cli_blk = cli.isel({dim_t: sl}).load()

    u_p  = u_blk - u_blk.mean(dim=[dim_y, dim_x])
    w_p  = w_blk - w_blk.mean(dim=[dim_y, dim_x])
    qc   = clw_blk + cli_blk

    flux_loc = rho0_da * u_p * w_p  # (bloc, z, y, x)

    m_conv = (w_p >  SEUIL_W) & (qc >  SEUIL_QC)
    m_sec  = (w_p >  SEUIL_W) & (qc <= SEUIL_QC)
    m_sub  = (w_p < -SEUIL_W)

    # Moyenne sur (t, y, x) — valeur nulle hors masque
    flux_conv_sum += flux_loc.where(m_conv, 0.0).mean(dim=[dim_t, dim_y, dim_x]).values
    flux_sec_sum  += flux_loc.where(m_sec,  0.0).mean(dim=[dim_t, dim_y, dim_x]).values
    flux_sub_sum  += flux_loc.where(m_sub,  0.0).mean(dim=[dim_t, dim_y, dim_x]).values
    flux_tot2_sum += flux_loc.mean(dim=[dim_t, dim_y, dim_x]).values
    n_cond += 1  # on accumule des moyennes, pas des sommes pondérées
    print(f'  t={t0:03d}–{t1-1:03d}')

flux_conv = flux_conv_sum / n_cond
flux_sec  = flux_sec_sum  / n_cond
flux_sub  = flux_sub_sum  / n_cond
flux_tot2 = flux_tot2_sum / n_cond
flux_rest = flux_tot2 - (flux_conv + flux_sec + flux_sub)

tend_conv = tendance(flux_conv)
tend_sec  = tendance(flux_sec)
tend_sub  = tendance(flux_sub)
tend_rest = tendance(flux_rest)
tend_tot2 = tendance(flux_tot2)

print('Flux conditionnels calculés.')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8), sharey=True)

kw = dict(tot  = dict(color='black',     lw=2.5),
          conv = dict(color='crimson',   lw=1.8, linestyle='--'),
          sec  = dict(color='darkorange',lw=1.8, linestyle='--'),
          sub  = dict(color='royalblue', lw=1.8, linestyle=':'),
          rest = dict(color='grey',      lw=1.5, alpha=0.7))

ax1.plot(flux_tot2, alt, label='Total',                   **kw['tot'])
ax1.plot(flux_conv, alt, label='Convection nuageuse',      **kw['conv'])
ax1.plot(flux_sec,  alt, label='Thermiques secs (CBL)',    **kw['sec'])
ax1.plot(flux_sub,  alt, label='Subsidence',               **kw['sub'])
ax1.plot(flux_rest, alt, label='Petite turbulence (reste)',**kw['rest'])
ax1.axvline(0, color='grey', alpha=0.4)
ax1.set_title('A. Flux ρ₀⟨u′w′⟩ par régime', fontweight='bold')
ax1.set_xlabel('Flux (kg m⁻¹ s⁻²)')
ax1.set_ylabel('Altitude (m)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

sc = 1e5
ax2.plot(tend_tot2 * sc, alt, label='Total',                   **kw['tot'])
ax2.plot(tend_conv * sc, alt, label='Convection nuageuse',      **kw['conv'])
ax2.plot(tend_sec  * sc, alt, label='Thermiques secs (CBL)',    **kw['sec'])
ax2.plot(tend_sub  * sc, alt, label='Subsidence',               **kw['sub'])
ax2.plot(tend_rest * sc, alt, label='Petite turbulence (reste)',**kw['rest'])
ax2.axvline(0, color='grey', alpha=0.4)
ax2.set_title('B. Tendance vent zonal par régime\n'
              r'$-\frac{1}{\rho_0}\frac{\partial}{\partial z}'
              r'(\rho_0\overline{u\'w\'})$  (×10⁻⁵ m/s²)', fontweight='bold')
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan conditionnel de QdM — décomposition par régime turbulent',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Bilan de QdM par région (sec / humide)

On applique le masque 2D (y, x) défini au §5 pour calculer  
les flux et tendances séparément dans chaque type de colonne.


In [ ]:
flux_h_sum = np.zeros(len(alt))
flux_s_sum = np.zeros(len(alt))
n_reg = 0

for t0 in range(t_stat, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    u_blk = u.isel({dim_t: sl}).load()
    w_blk = w.isel({dim_t: sl}).load()

    u_p = u_blk - u_blk.mean(dim=[dim_y, dim_x])
    w_p = w_blk - w_blk.mean(dim=[dim_y, dim_x])
    flux_loc = (rho0_da * u_p * w_p).values  # numpy (bloc, z, y, x)

    # Sélection des colonnes selon le masque (axe y et x)
    # mh/ms : (y, x) — on sélectionne les points valides sur les deux derniers axes
    flux_h_sum += flux_loc[:, :, mh].mean(axis=(0, 2))
    flux_s_sum += flux_loc[:, :, ms].mean(axis=(0, 2))
    n_reg += 1
    print(f'  t={t0:03d}–{t1-1:03d}')

flux_humide = flux_h_sum / n_reg
flux_sec_r  = flux_s_sum / n_reg

tend_humide = tendance(flux_humide)
tend_sec_r  = tendance(flux_sec_r)

print('Flux par région calculés.')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

ax1.plot(flux_uw,    alt, 'k-',  lw=2.5, label='Global')
ax1.plot(flux_humide,alt, color='royalblue',    lw=2, linestyle='--', label='Humide')
ax1.plot(flux_sec_r, alt, color='saddlebrown',  lw=2, linestyle=':',  label='Sec')
ax1.axvline(0, color='grey', alpha=0.4)
ax1.set_title('Flux ρ₀⟨u′w′⟩ — global vs régions', fontweight='bold')
ax1.set_xlabel('Flux (kg m⁻¹ s⁻²)')
ax1.set_ylabel('Altitude (m)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(tend_u_glob  * 1e5, alt, 'k-',  lw=2.5, label='Global')
ax2.plot(tend_humide  * 1e5, alt, color='royalblue',   lw=2, linestyle='--', label='Humide')
ax2.plot(tend_sec_r   * 1e5, alt, color='saddlebrown', lw=2, linestyle=':',  label='Sec')
ax2.axvline(0, color='grey', alpha=0.4)
ax2.set_title('Tendance vent zonal — global vs régions\n(×10⁻⁵ m/s²)', fontweight='bold')
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan de QdM : régions sèches vs humides', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Évolution temporelle du flux de Reynolds intégré

Le flux ρ₀⟨u′w′⟩ intégré verticalement sur la troposphère (0–15 km)  
à chaque pas de temps permet de suivre la maturation de la simulation.


In [ ]:
idx_tropo = np.searchsorted(alt, 15000)  # indice ~15 km
alt_tropo = alt[:idx_tropo]

flux_int_time = []

for t0 in range(0, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    u_blk = u.isel({dim_t: sl}).load()
    w_blk = w.isel({dim_t: sl}).load()

    u_p = u_blk - u_blk.mean(dim=[dim_y, dim_x])
    w_p = w_blk - w_blk.mean(dim=[dim_y, dim_x])

    flux_prof = (rho0_da * u_p * w_p).mean(dim=[dim_y, dim_x])  # (bloc, z)
    # Intégration verticale jusqu'à ~15 km
    flux_int = np.trapz(
        flux_prof.values[:, :idx_tropo], alt_tropo, axis=1
    )  # (bloc,)
    flux_int_time.append(flux_int)

flux_int_time = np.concatenate(flux_int_time)  # (n_times,)
time_days = u[dim_t].values.astype('float64') / (1e9 * 3600 * 24)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time_days, flux_int_time, color='black', lw=1.2)
ax.axvline(time_days[t_stat], color='red', lw=1.2, linestyle='--',
           label=f'Début état stationnaire (t={time_days[t_stat]:.1f} j)')
ax.set_xlabel('Temps (jours)')
ax.set_ylabel('∫ ρ₀⟨u′w′⟩ dz  (kg/s²)')
ax.set_title('Évolution temporelle du flux de Reynolds intégré (0–15 km)',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Notes méthodologiques

### Gestion mémoire
Tous les fichiers 3D sont ouverts en mode **lazy** (`chunks={'time': BLOC}`).  
Les calculs se font par blocs de `BLOC` pas de temps : seul le bloc courant est  
en RAM à un instant donné. La variable `BLOC` (cellule §2) est le seul paramètre  
à ajuster selon la RAM disponible (diminuer à 5 si nécessaire).

La PRW est lue depuis son fichier 2D natif : chargement total en RAM possible  
car ce fichier est léger (pas de dimension verticale).

### Densité de référence ρ₀
$$\rho_0 = \frac{p}{R_d \, T_v}, \qquad T_v = T \cdot \frac{1 + q_v/\varepsilon}{1 + q_v}, \qquad \varepsilon = R_d/R_v$$
ρ₀ est moyenné sur le domaine horizontal et l'état stationnaire → profil 1D(z).

### Flux de Reynolds pondéré
Le flux est exprimé sous forme ρ₀⟨u′w′⟩ [kg m⁻¹ s⁻²] et non ⟨u′w′⟩ [m² s⁻²],  
ce qui est nécessaire pour que la divergence verticale donne directement  
une tendance en m/s² dans l'équation de QdM en coordonnées z.

### Seuil sec / humide
Initialisé à la médiane de la PRW. À affiner selon la forme de la distribution  
(creux entre les deux modes si agrégation présente).
